# Customer Churn Prediction
Predicting which telecom customers are likely to stop using the service.

**Dataset:** Telco Customer Churn (Kaggle) | **Author:** Manku Deepika

## Step 1: Load and Clean Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna().drop("customerID", axis=1)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(df.shape)
df.head()

## Step 2: Exploratory Data Analysis (EDA)

In [ ]:
print(df["Churn"].value_counts(normalize=True) * 100)
sns.countplot(x="Churn", data=df)
plt.title("Churn count (0 = stayed, 1 = left)")
plt.show()

In [ ]:
cat_cols = ["Contract", "InternetService", "PaymentMethod",
            "TechSupport", "OnlineSecurity", "gender", "SeniorCitizen"]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, col in zip(axes.flatten(), cat_cols):
    df.groupby(col)["Churn"].mean().mul(100).plot(kind="bar", ax=ax)
    ax.set_title(f"Churn % by {col}")
    ax.set_ylabel("Churn %")
    ax.tick_params(axis="x", rotation=30)
axes.flatten()[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue="Churn", kde=True, ax=axes[0, i])
    sns.boxplot(data=df, x="Churn", y=col, ax=axes[1, i])
    axes[0, i].set_title(f"{col} distribution")
    axes[1, i].set_title(f"{col} by churn")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(df[num_cols + ["Churn"]].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation heatmap")
plt.show()

### EDA findings
- Month-to-month contract customers churn far more than 1-year/2-year customers
- Fiber optic customers churn more than DSL
- Customers without OnlineSecurity/TechSupport churn more
- Electronic check payers churn more
- New customers (low tenure) churn the most

## Step 3: Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

df = df.replace({"No internet service": "No", "No phone service": "No"})

X = df.drop("Churn", axis=1)
y = df["Churn"]
X = pd.get_dummies(X, drop_first=True, dtype=int)
print("Features after encoding:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train churn %:", round(y_train.mean() * 100, 1))
print("Test churn %:", round(y_test.mean() * 100, 1))

In [ ]:
scaler = StandardScaler()
X_train_sc, X_test_sc = X_train.copy(), X_test.copy()
X_train_sc[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_sc[num_cols] = scaler.transform(X_test[num_cols])

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("\nAfter SMOTE:\n", y_train_sm.value_counts())

## Step 4: Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=4,
                             eval_metric="logloss", random_state=42),
}

results = []
trained = {}
for name, model in models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    trained[name] = model
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    })

results_df = pd.DataFrame(results).set_index("Model").round(3)
results_df

## Step 5: Evaluation

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, model) in zip(axes, trained.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test_sc, y_test, display_labels=["Stayed", "Churned"],
        cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 6))
for name, model in trained.items():
    RocCurveDisplay.from_estimator(model, X_test_sc, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random guess")
ax.set_title("ROC curves: churn models")
ax.legend(loc="lower right")
plt.show()

In [ ]:
xgb_imp = pd.Series(trained["XGBoost"].feature_importances_, index=X.columns)
xgb_imp.sort_values().tail(10).plot(kind="barh", figsize=(8, 5))
plt.title("Top 10 features: XGBoost importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## Conclusion

**Best model: Logistic Regression** — best ROC-AUC (0.828) and recall (0.743), and simplest to explain to a business team.

**Business recommendations:**
- Offer incentives to move month-to-month customers onto longer contracts
- Bundle free online security / tech support for new customers
- Focus retention efforts on customers in their first 12 months